In [ ]:
# Notebook top: configure LM Studio URL and attach runtime patches
import os, sys
workspace = '/home/woogaxon/WORK/DEV'
if workspace not in sys.path:
    sys.path.insert(0, workspace)
# Set the LM Studio base URL for this kernel (no API key required)
os.environ['LM_STUDIO_API_URL'] = 'http://10.81.28.113:1234'
try:
    import lmstudio_adapter
    lmstudio_adapter.attach_runtime_patches(globals())
    print('LM Studio URL set and runtime patches attached; module=', getattr(lmstudio_adapter, '__file__', None))
except Exception as e:
    print('Failed to attach lmstudio_adapter runtime patches:', e)
    raise

In [ ]:
# Example Pydantic model used by the notebook's structured-output tests
from pydantic import BaseModel
class MeetingSummary(BaseModel):
    summary: str
    action_items: list[str] = []
print('Inserted example MeetingSummary model')

In [ ]:
# Ensure an `llm` variable exists for downstream cells (module-backed instance if available)
if 'llm' not in globals() or llm is None:
    try:
        if 'LMStudioChat' in globals() and LMStudioChat is not None:
            llm = LMStudioChat()
            print('Created module-backed llm instance for notebook, model=', getattr(llm, 'model', None))
        else:
            llm = None
            print('LMStudioChat not available; llm set to None')
    except Exception as e:
        print('Failed to create llm instance:', e)
        llm = None

In [ ]:
# Import module-backed implementations and attach runtime helpers to this notebook globals
import sys, os
workspace = '/home/woogaxon/WORK/DEV'
if workspace not in sys.path:
    sys.path.insert(0, workspace)
try:
    import lmstudio_adapter
    from lmstudio_adapter import LMStudioChat, StructuredLLM, attach_runtime_patches
    _mods = attach_runtime_patches(globals())
    print('Imported lmstudio_adapter; file=', getattr(lmstudio_adapter, '__file__', None))
    print('Attached runtime patches:', list(_mods.keys()))
except Exception as e:
    print('Failed to import lmstudio_adapter:', e)
    raise

In [ ]:
# LMStudioChatRuntime/LMStudioChatSafe aliases now point to module-backed LMStudioChat.
# This cell is a lightweight compatibility shim; the implementation is in lmstudio_adapter.py.
try:
    LMStudioChatRuntime = LMStudioChat
    LMStudioChatSafe = LMStudioChat
except NameError:
    LMStudioChatRuntime = None
    LMStudioChatSafe = None
# lazily create llm for interactive convenience if missing
if LMStudioChatSafe is not None and 'llm' not in globals():
    try:
        llm = LMStudioChatSafe()
        print('Created module-backed llm instance, model=', llm.model)
    except Exception as e:
        print('Failed to instantiate module-backed LMStudioChat:', e)

LMStudioChatSafe instantiated, model= exaone-3.5-2.4b-instruct


In [ ]:
# Verify that current llm instance supports with_structured_output and that Summary_llm works (assistant-inserted)
import traceback, json
# Ensure MeetingSummary model exists (be defensive in headless runs)
if 'MeetingSummary' not in globals():
    try:
        from pydantic import BaseModel
        class MeetingSummary(BaseModel):
            summary: str
            action_items: list[str] = []
        print('Defined fallback MeetingSummary model')
    except Exception:
        print('Pydantic not available; MeetingSummary will remain undefined')
# ensure an llm exists for verification
if 'llm' not in globals() or llm is None:
    try:
        if 'LMStudioChat' in globals() and LMStudioChat is not None:
            llm = LMStudioChat()
            print('Created llm instance for verification, model=', getattr(llm, 'model', None))
        else:
            print('LMStudioChat not available; skipping verification')
    except Exception as e:
        print('Failed to create llm for verification:', e)
        traceback.print_exc()

if 'llm' in globals() and llm is not None:
    try:
        Summary_llm = llm.with_structured_output(MeetingSummary)
        print('with_structured_output call succeeded, Summary_llm type=', type(Summary_llm))
        prompt = '다음 회의록을 읽고 JSON 형식으로 요약하고 action_items를 추출하세요. 짧게.'
        if 'sample_transcript' in globals():
            prompt += '\n\n' + sample_transcript
        res = Summary_llm.invoke(prompt, system_prompt='You are concise. Return valid JSON only.', max_tokens=512)
        print('\nInvocation returned type:', type(res))
        if hasattr(res, 'model_dump_json'):
            print('\nParsed JSON:\n', res.model_dump_json())
        elif hasattr(res, 'json'):
            print('\nParsed JSON:\n', res.json())
        else:
            print('\nRaw parsed:\n', json.dumps(res, ensure_ascii=False, indent=2))
    except Exception:
        print('Verification failed:')
        traceback.print_exc()
else:
    print('Skipping verification because llm is not available')
# Note: runtime patching of StructuredLLM.invoke is handled by the module-backed implementation
# The module's attach_runtime_patches(...) already ensures StructuredLLM and LMStudioChat are wired in notebooks.
try:
    import lmstudio_adapter
    print('lmstudio_adapter loaded; module file=', getattr(lmstudio_adapter, '__file__', None))
    print('If you need to force-patch invoke, use lmstudio_adapter.StructuredLLM.invoke = your_func')
except Exception as e:
    print('Could not import lmstudio_adapter in this kernel:', e)


llm type: <class '__main__.LMStudioChatRuntime'>
with_structured_output call succeeded, Summary_llm type= <class '__main__.StructuredLLM'>

Invocation returned type: <class '__main__.MeetingSummary'>

Parsed JSON:
 {"summary":"지난 회의에서 김민준이 사용자 피드백 분석 결과를 공유했으며, 로그인 절차 개선 필요성에 동의했다. 박서준은 UI/UX 초안을 5월 6일까지 제작하고, 이지혜는 경쟁사 앱 로그인 방식 리서치를 이번 주 금요일까지 완료하도록 요청받았다. 다음 회의 전까지 각자의 업무 마무리를 목표로 설정했다.","action_items":[{"assignee":"박서준","task":"UI/UX 로그인 초안 디자인 완성 (5월 6일)","due_date":null},{"assignee":"이지혜","task":"경쟁사 앱 로그인 방식 리서치 완료 (5월 2일)","due_date":null}]}

Invocation returned type: <class '__main__.MeetingSummary'>

Parsed JSON:
 {"summary":"지난 회의에서 김민준이 사용자 피드백 분석 결과를 공유했으며, 로그인 절차 개선 필요성에 동의했다. 박서준은 UI/UX 초안을 5월 6일까지 제작하고, 이지혜는 경쟁사 앱 로그인 방식 리서치를 이번 주 금요일까지 완료하도록 요청받았다. 다음 회의 전까지 각자의 업무 마무리를 목표로 설정했다.","action_items":[{"assignee":"박서준","task":"UI/UX 로그인 초안 디자인 완성 (5월 6일)","due_date":null},{"assignee":"이지혜","task":"경쟁사 앱 로그인 방식 리서치 완료 (5월 2일)","due_date":null}]}


In [ ]:
# Runtime patch: replace StructuredLLM.invoke to force Pydantic v2 usage at runtime
import types, traceback

def _invoke_v2(self, input, system_prompt: str = None, max_tokens: int = 512):
    resp = self._parent.invoke(input, system_prompt=system_prompt, max_tokens=max_tokens)
    text = getattr(resp, 'content', None) or str(resp)
    # 1) try parser
    if self._parser is not None:
        try:
            if hasattr(self._parser, 'parse'):
                return self._parser.parse(text)
            if hasattr(self._parser, 'parse_text'):
                return self._parser.parse_text(text)
        except Exception:
            pass
    # 2) pydantic v2
    try:
        if hasattr(self._model, 'model_validate_json'):
            try:
                return self._model.model_validate_json(text)
            except Exception:
                pass
    except Exception:
        pass
    # 3) extract json and use model_validate
    try:
        j = None
        import re, json as _json
        m = re.search(r"(?s)(\{.*\}|\[.*\])", text)
        if m:
            try:
                j = _json.loads(m.group(0))
            except Exception:
                try:
                    j = _json.loads(m.group(0).replace("'", '"'))
                except Exception:
                    j = None
        if j is not None:
            if hasattr(self._model, 'model_validate'):
                try:
                    return self._model.model_validate(j)
                except Exception:
                    return j
            # fallback legacy
            if hasattr(self._model, 'parse_obj'):
                try:
                    return self._model.parse_obj(j)
                except Exception:
                    return j
    except Exception:
        pass
    raise RuntimeError('Structured parsing failed. Output:\n' + text)

# bind to class
# NOTE: Direct assignment to StructuredLLM.invoke is unsafe in fresh kernels where StructuredLLM may not be defined.
# The module lmstudio_adapter provides the canonical implementation and attach_runtime_patches to handle this.
print('Skipped unsafe direct patch; use lmstudio_adapter.attach_runtime_patches(globals()) or assign lmstudio_adapter.StructuredLLM.invoke explicitly if needed')

# run the existing smoke test cell to verify
try:
    get_ipython().run_cell("# 통합 스모크 테스트: LMStudioChat + with_structured_output + MeetingSummary 파싱\nimport traceback, json\nprint('Kernel llm variable type:', type(llm))\n# ensure llm is an instance of a class with with_structured_output\ntry:\n    if not hasattr(llm, 'with_structured_output'):\n        print('Current llm lacks with_structured_output; creating a fresh LMStudioChat instance')\n        llm = LMStudioChat()\n    # create structured wrapper\n    Summary_llm = llm.with_structured_output(MeetingSummary)\n    print('Summary_llm created:', type(Summary_llm))\n    # prepare prompt\n    prompt = '다음 회의록을 읽고 한 문장 요약과 action_items를 JSON으로 출력하세요.'\n    if 'sample_transcript' in globals():\n        prompt += '\\n\\n' + sample_transcript\n    res = Summary_llm.invoke(prompt, system_prompt='간결한 한국어로, JSON만 반환하세요.', max_tokens=512)\n    print('\\nResult type:', type(res))\n    if hasattr(res, 'model_dump_json'):\n        print('\\nParsed Pydantic JSON:\\n', res.model_dump_json())\n    elif hasattr(res, 'json'):\n        print('\\nParsed Pydantic JSON:\\n', res.json())\n    else:\n        print('\\nParsed raw:\\n', json.dumps(res, ensure_ascii=False, indent=2))\nexcept Exception:\n    print('Smoke test failed:')\n    traceback.print_exc()")
except Exception:
    print('Failed to re-run smoke test automatically')


StructuredLLM.invoke has been patched at runtime to use Pydantic v2 methods (if available)
Kernel llm variable type: <class '__main__.LMStudioChatRuntime'>
Summary_llm created: <class '__main__.StructuredLLM'>

Result type: <class 'dict'>

Parsed raw:
 {
  "summary": "김민준이 사용자 피드백 분석 결과를 공유하고, 박서준에게 로그인 UI/UX 개선 초안 제작을 요청했으며, 이지혜에게 경쟁사 앱 리서치를 지시했다. 모든 작업은 다음 주 회의 전까지 완료하기로 했다.",
  "action_items": [
    {
      "responsible": "박서준",
      "task": "로그인 UI/UX 개선 초안 제작",
      "deadline": "2025-04-30"
    },
    {
      "responsible": "이지혜",
      "task": "경쟁사 앱 로그인 방식 리서치",
      "deadline": "2025-05-02"
    }
  ]
}

Result type: <class 'dict'>

Parsed raw:
 {
  "summary": "김민준이 사용자 피드백 분석 결과를 공유하고, 박서준에게 로그인 UI/UX 개선 초안 제작을 요청했으며, 이지혜에게 경쟁사 앱 리서치를 지시했다. 모든 작업은 다음 주 회의 전까지 완료하기로 했다.",
  "action_items": [
    {
      "responsible": "박서준",
      "task": "로그인 UI/UX 개선 초안 제작",
      "deadline": "2025-04-30"
    },
    {
      "responsible": "이지혜",
      "task": "경쟁사 앱 로그인 방식 리서치",
      "dead

In [49]:
# Diagnostic: locate StructuredLLM cell in notebook and print limited source for inspection
import json, os
nb_path = '/home/woogaxon/WORK/DEV/colabNotebooks/1.LangChainRAG4LMS.ipynb'
print('Reading notebook:', nb_path)
try:
    with open(nb_path, 'r', encoding='utf-8') as f:
        nb = json.load(f)
    found = False
    for idx, cell in enumerate(nb.get('cells', []), start=1):
        src = cell.get('source', '')
        if isinstance(src, list):
            joined = ''.join(src)
        else:
            joined = str(src)
        if 'class StructuredLLM' in joined or 'def _extract_first_json' in joined and 'StructuredLLM' in joined:
            print(f'Found StructuredLLM in notebook cell #{idx}')
            # print limited amount
            lines = joined.splitlines()
            to_print = '\n'.join(lines[:80])
            snippet = to_print[:2000]
            print('\n--- begin snippet (first 80 lines / 2000 chars) ---\n')
            print(snippet)
            print('\n--- end snippet ---\n')
            found = True
            break
    if not found:
        print('StructuredLLM definition not found in notebook file')
except Exception as e:
    print('Failed to read notebook file:', repr(e))
    raise


Reading notebook: /home/woogaxon/WORK/DEV/colabNotebooks/1.LangChainRAG4LMS.ipynb
Found StructuredLLM in notebook cell #4

--- begin snippet (first 80 lines / 2000 chars) ---

# Inserted cell: robust StructuredLLM and monkey-patch for LMStudioChat.with_structured_output
import re
import json
from typing import Any

# helper: extract first JSON object/array from text
def _extract_first_json(text: str) -> Any:
    # try to find a JSON object or array with a simple regex
    # This finds the first {...} or [...] block with a greedy match
    obj_re = re.search(r"(?s)(\{.*\}|\[.*\])", text)
    if not obj_re:
        return None
    s = obj_re.group(0)
    try:
        return json.loads(s)
    except Exception:
        # fallback: try to fix common single-quotes -> double-quotes
        try:
            fixed = s.replace("'", '"')
            return json.loads(fixed)
        except Exception:
            return None

# Structured wrapper class
class StructuredLLM:
    def __init__(self, pa